In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1b_MvBef6tItbDXaOe1gzK-9-gRUhhxnI", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/05_00_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Prompt Design Capstone: Composing Techniques

*Part 5 of the Vizuara series on Prompt Design Principles*
*Estimated time: 70 minutes*

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/prompt-design-principles/practice/5/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Motivation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_01_motivation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

You have now learned four powerful prompt design techniques: system prompts at the right altitude, few-shot examples, chain-of-thought reasoning, and structured output formats. Each one is useful on its own. But here is the real secret — the professionals do not use these techniques in isolation. They **compose** them, layering multiple techniques together in carefully orchestrated pipelines.

Think of it this way. A chef who knows how to chop, saute, braise, and plate has four skills. But a great meal requires combining all four in the right order, at the right time. The same is true for prompt design. Knowing when to use which technique — and how to chain them together — is the difference between a clever demo and a production-ready system.

In this capstone notebook, we will:
- Build a decision framework for choosing the right technique
- Understand the mathematics of combining techniques (reliability analysis and token budgets)
- Construct a complete code review pipeline using ALL five techniques together
- Build an end-to-end document analysis pipeline with gate checks between stages
- Visualize pipeline performance and technique interactions

Let us start by building the intuition for composition.

In [ ]:
#@title 🎧 Listen: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_02_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Setup — install dependencies
!pip install -q anthropic matplotlib pandas numpy

import os
import json
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from anthropic import Anthropic

client = Anthropic()

def query_llm(system_prompt, user_message, model="claude-sonnet-4-20250514", max_tokens=2048):
    """Send a message to Claude with a given system prompt and return the response text."""
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text.strip()

def query_llm_json(system_prompt, user_message, model="claude-sonnet-4-20250514", max_tokens=2048):
    """Send a message to Claude and parse the response as JSON."""
    raw = query_llm(system_prompt, user_message, model, max_tokens)
    # Extract JSON from response (handles cases where model wraps in markdown)
    if "```json" in raw:
        raw = raw.split("```json")[1].split("```")[0]
    elif "```" in raw:
        raw = raw.split("```")[1].split("```")[0]
    return json.loads(raw.strip())

print("Setup complete!")

In [ ]:
#@title 🎧 Listen: Decision Framework Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_03_decision_framework_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. The Decision Framework: When to Use Which Technique

Before we compose techniques, we need a clear mental model for when each one is appropriate. Let us build this decision framework from first principles.

### The Technique Selection Criteria

Each technique addresses a different failure mode:

| Technique | Solves This Problem | Best When |
|---|---|---|
| **System Prompt** (Right Altitude) | Model does not know its role or constraints | You need consistent behavior across many inputs |
| **Few-Shot Examples** | Model does not know your format or labeling scheme | Output format matters, or the task is ambiguous |
| **Chain-of-Thought** | Model jumps to conclusions on complex reasoning | Multi-step logic, math, or analysis is required |
| **Structured Output** | Model returns free-form text when you need data | Downstream code must parse the response |
| **Prompt Chaining** | A single prompt cannot handle the full task reliably | Task has multiple distinct stages with different requirements |

### Think About This

Consider a task: "Given a customer support email, classify it by department, extract the order number, summarize the issue, and suggest a response."

Should you handle this in one prompt or four? What happens if the classification is wrong — does the summary still make sense? What format should the output be in?

The answers to these questions tell you exactly which techniques to compose.

In [ ]:
#@title 🎧 Listen: Decision Framework Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_04_decision_framework_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Decision Framework: A practical technique selector

def recommend_techniques(task_description):
    """Use Claude to recommend which prompt techniques to apply for a given task."""
    system_prompt = """You are a prompt engineering advisor. Given a task description,
recommend which prompt design techniques to use and WHY.

Techniques available:
1. System Prompt (role, constraints, behavioral guidelines)
2. Few-Shot Examples (input-output demonstrations)
3. Chain-of-Thought (step-by-step reasoning)
4. Structured Output (JSON/schema format)
5. Prompt Chaining (multi-step pipeline with gate checks)

Respond in this exact JSON format:
{
    "techniques": ["technique1", "technique2"],
    "reasoning": "Why these techniques in this combination",
    "composition_order": "How to layer them together",
    "risk_without": "What goes wrong if you skip any of these"
}"""

    return query_llm_json(system_prompt, f"Task: {task_description}")

# Test the decision framework
task = "Analyze a legal contract to extract key clauses, identify potential risks, and generate a plain-English summary for a non-lawyer."
recommendation = recommend_techniques(task)

print("Task:", task)
print("\nRecommended techniques:", recommendation["techniques"])
print("\nReasoning:", recommendation["reasoning"])
print("\nComposition order:", recommendation["composition_order"])
print("\nRisk if skipped:", recommendation["risk_without"])

In [ ]:
#@title 🎧 Listen: Decision Framework Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_05_decision_framework_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: Decision tree for technique selection
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Prompt Technique Decision Tree', fontsize=16, fontweight='bold', pad=20)

# Color palette
colors = {
    'question': '#E3F2FD',
    'system': '#C8E6C9',
    'fewshot': '#FFF9C4',
    'cot': '#F8BBD0',
    'structured': '#D1C4E9',
    'chaining': '#FFE0B2',
    'border': '#37474F'
}

def draw_box(ax, x, y, w, h, text, color, fontsize=9):
    """Draw a rounded box with centered text."""
    box = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                   boxstyle="round,pad=0.15",
                                   facecolor=color, edgecolor=colors['border'],
                                   linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', wrap=True,
            bbox=dict(boxstyle='round,pad=0.1', facecolor=color, edgecolor='none'))

def draw_arrow(ax, x1, y1, x2, y2, label="", color='#37474F'):
    """Draw an arrow between two points with an optional label."""
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx + 0.15, my + 0.1, label, fontsize=8, color=color, fontstyle='italic')

# Root question
draw_box(ax, 7, 9.2, 5.5, 0.7, "Does the model need a consistent\nrole across many inputs?", colors['question'])
draw_arrow(ax, 5.5, 8.85, 3.5, 8.25, "Yes")
draw_arrow(ax, 8.5, 8.85, 10.5, 8.25, "No")

# System prompt node
draw_box(ax, 3.5, 7.9, 3.0, 0.6, "Add SYSTEM PROMPT\n(Right Altitude)", colors['system'])

# No-system branch question
draw_box(ax, 10.5, 7.9, 3.5, 0.6, "Is the output format\nambiguous or critical?", colors['question'])

# Second level
draw_arrow(ax, 3.5, 7.6, 3.5, 7.0, "Then ask:")
draw_box(ax, 3.5, 6.6, 3.5, 0.6, "Is the output format\nambiguous or critical?", colors['question'])

draw_arrow(ax, 9.5, 7.6, 7.5, 7.0, "Yes")
draw_arrow(ax, 11.5, 7.6, 12.5, 7.0, "No")

draw_box(ax, 7.5, 6.6, 3.0, 0.6, "Add FEW-SHOT EXAMPLES\n(3-5 demonstrations)", colors['fewshot'])
draw_box(ax, 12.5, 6.6, 2.5, 0.6, "Zero-shot may\nbe sufficient", colors['question'])

# Few-shot from left branch
draw_arrow(ax, 2.5, 6.3, 1.5, 5.7, "Yes")
draw_arrow(ax, 4.5, 6.3, 5.5, 5.7, "No")
draw_box(ax, 1.5, 5.3, 3.0, 0.6, "Add FEW-SHOT EXAMPLES\n(3-5 demonstrations)", colors['fewshot'])

# Reasoning question
draw_box(ax, 5.5, 5.3, 3.5, 0.6, "Does the task require\nmulti-step reasoning?", colors['question'])

draw_arrow(ax, 1.5, 5.0, 1.5, 4.3, "Then ask:")
draw_box(ax, 1.5, 3.9, 3.5, 0.6, "Does the task require\nmulti-step reasoning?", colors['question'])

draw_arrow(ax, 4.5, 5.0, 4.0, 4.3, "Yes")
draw_arrow(ax, 6.5, 5.0, 8.0, 4.3, "No")

draw_box(ax, 4.0, 3.9, 2.8, 0.6, "Add CHAIN-OF-THOUGHT\n(step-by-step)", colors['cot'])

# Structured output question
draw_box(ax, 8.0, 3.9, 3.5, 0.6, "Must downstream code\nparse the output?", colors['question'])
draw_arrow(ax, 7.0, 3.6, 6.0, 2.9, "Yes")
draw_arrow(ax, 9.0, 3.6, 10.0, 2.9, "No")
draw_box(ax, 6.0, 2.5, 3.0, 0.6, "Add STRUCTURED OUTPUT\n(JSON schema)", colors['structured'])

# CoT left branch
draw_arrow(ax, 0.7, 3.6, 0.7, 2.9, "Yes")
draw_arrow(ax, 2.3, 3.6, 3.0, 2.9, "No")
draw_box(ax, 0.7, 2.5, 2.2, 0.6, "Add CHAIN-OF-\nTHOUGHT", colors['cot'])

# Final chaining question
draw_box(ax, 7, 1.3, 5.5, 0.7, "Is the full task too complex for\none prompt? Multiple distinct stages?", colors['question'])
draw_arrow(ax, 6.0, 2.2, 6.0, 1.7, "Then ask:")
draw_arrow(ax, 5.5, 0.95, 4.0, 0.45, "Yes")
draw_arrow(ax, 8.5, 0.95, 10.0, 0.45, "No")
draw_box(ax, 4.0, 0.2, 3.5, 0.5, "Add PROMPT CHAINING\n(pipeline + gate checks)", colors['chaining'])
draw_box(ax, 10.0, 0.2, 2.5, 0.5, "Single composed\nprompt is fine", colors['system'])

plt.tight_layout()
plt.show()

print("Key insight: You traverse this tree top-to-bottom, collecting techniques.")
print("Most real-world tasks end up needing 3-4 techniques composed together.")

In [ ]:
#@title 🎧 Listen: Math Intro Reliability
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_06_math_intro_reliability.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics of Composition

Now let us formalize what happens when we combine techniques. This gives us a principled way to reason about pipeline design.

### Reliability Analysis: Chained Steps

When you chain $n$ prompt steps in a pipeline, the overall reliability is the product of individual step reliabilities:

$$R_{\text{pipeline}} = \prod_{i=1}^{n} R_i$$

where $R_i$ is the reliability (probability of correct output) of step $i$.

Let us plug in some simple numbers. Suppose you have a 3-step pipeline:
- Step 1 (extraction): $R_1 = 0.95$
- Step 2 (classification): $R_2 = 0.92$
- Step 3 (summarization): $R_3 = 0.90$

Without gate checks: $R_{\text{pipeline}} = 0.95 \times 0.92 \times 0.90 = 0.786$

That is only 78.6% reliability for a 3-step pipeline, even though each step is above 90%. This is the **compound reliability problem** — and it is why gate checks matter.

### Gate Checks Improve Reliability

A gate check after step $i$ catches errors with probability $g_i$ and retries. With gate checks:

$$R_{\text{gated}} = \prod_{i=1}^{n} \left[R_i + (1 - R_i) \cdot g_i \cdot R_i\right]$$


In [ ]:
#@title 🎧 Listen: Math Gates Tokens
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_07_math_gates_tokens.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

The term $(1 - R_i) \cdot g_i \cdot R_i$ represents: the probability of initial failure times the probability the gate catches it times the probability the retry succeeds.

Using the same numbers with gate checks that catch 80% of errors ($g_i = 0.8$):
- Step 1 gated: $0.95 + 0.05 \times 0.8 \times 0.95 = 0.95 + 0.038 = 0.988$
- Step 2 gated: $0.92 + 0.08 \times 0.8 \times 0.92 = 0.92 + 0.059 = 0.979$
- Step 3 gated: $0.90 + 0.10 \times 0.8 \times 0.90 = 0.90 + 0.072 = 0.972$

$R_{\text{gated}} = 0.988 \times 0.979 \times 0.972 = 0.940$

Gate checks boosted overall reliability from 78.6% to 94.0%. This is exactly what we want.

### Token Budget Formulation

When composing techniques, each one consumes tokens from your budget:

$$T_{\text{total}} = T_{\text{system}} + k \cdot T_{\text{example}} + T_{\text{cot\_instruction}} + T_{\text{schema}} + T_{\text{input}}$$

where $k$ is the number of few-shot examples and $T_{\text{example}}$ is the average token count per example. The model's context window sets the upper bound. Let us compute a concrete budget.

Suppose our context window is 200,000 tokens and we allocate:
- System prompt: $T_{\text{system}} = 300$ tokens
- 5 few-shot examples at 150 tokens each: $k \cdot T_{\text{example}} = 5 \times 150 = 750$ tokens
- CoT instruction: $T_{\text{cot\_instruction}} = 50$ tokens
- JSON schema: $T_{\text{schema}} = 100$ tokens
- Input document: $T_{\text{input}} = 2000$ tokens

Total prompt: $300 + 750 + 50 + 100 + 2000 = 3200$ tokens, leaving 196,800 for the response. Plenty of room. But if each example were 2000 tokens (long code examples) and we used 10 of them, the examples alone would consume 20,000 tokens. Budgeting matters.

In [ ]:
#@title 🎧 Listen: Math Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_08_math_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: Reliability with and without gate checks
n_steps = np.arange(1, 8)
base_reliability = 0.92  # Each step is 92% reliable
gate_catch_rate = 0.80

# Without gates
R_no_gate = base_reliability ** n_steps

# With gates
R_per_step_gated = base_reliability + (1 - base_reliability) * gate_catch_rate * base_reliability
R_with_gate = R_per_step_gated ** n_steps

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Reliability decay
axes[0].plot(n_steps, R_no_gate, 'o-', color='#E53935', linewidth=2.5,
             markersize=8, label='Without gate checks')
axes[0].plot(n_steps, R_with_gate, 's-', color='#43A047', linewidth=2.5,
             markersize=8, label='With gate checks (80% catch rate)')
axes[0].axhline(y=0.80, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
axes[0].fill_between(n_steps, R_no_gate, R_with_gate, alpha=0.15, color='green')
axes[0].set_xlabel('Number of Pipeline Steps', fontsize=12)
axes[0].set_ylabel('Overall Reliability', fontsize=12)
axes[0].set_title('Pipeline Reliability Decays Without Gates', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0.4, 1.02)
axes[0].grid(True, alpha=0.3)

# Annotate the gap at 5 steps
idx = 4  # 5 steps
gap = R_with_gate[idx] - R_no_gate[idx]
axes[0].annotate(f'+{gap:.1%}\nreliability',
                 xy=(5, (R_with_gate[idx] + R_no_gate[idx]) / 2),
                 fontsize=11, fontweight='bold', color='green',
                 ha='center')

# Plot 2: Token budget breakdown
techniques = ['System\nPrompt', 'Few-Shot\n(5 examples)', 'CoT\nInstruction', 'JSON\nSchema', 'Input\nDocument']
token_counts = [300, 750, 50, 100, 2000]
colors_budget = ['#66BB6A', '#FFA726', '#EF5350', '#AB47BC', '#42A5F5']

bars = axes[1].bar(techniques, token_counts, color=colors_budget, edgecolor='black', linewidth=1.2)
axes[1].set_ylabel('Token Count', fontsize=12)
axes[1].set_title('Token Budget Breakdown for Composed Prompt', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')

for bar, count in zip(bars, token_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 30,
                 f'{count}', ha='center', fontsize=11, fontweight='bold')

total = sum(token_counts)
axes[1].text(0.98, 0.95, f'Total: {total:,} tokens\n(1.6% of 200K context)',
             transform=axes[1].transAxes, fontsize=10, ha='right', va='top',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

plt.tight_layout()
plt.show()

print(f"Key insight: Gate checks recover {gap:.1%} reliability at 5 steps.")
print(f"Token overhead for all techniques: {total:,} tokens — a small fraction of modern context windows.")

In [ ]:
#@title 🎧 Listen: Code Review Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_09_code_review_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Building It — The Complete Code Review Pipeline

Now let us put everything together. We will build a code review pipeline that composes ALL five techniques:

1. **System prompt** (right altitude) for consistent reviewer behavior
2. **Few-shot examples** to teach the output format
3. **Chain-of-thought reasoning** for deep analysis
4. **Structured JSON output** for machine-readable results
5. **Prompt chaining with gate checks** for multi-stage review

This is where the real power of composition becomes visible.

### 4.1 Stage 1 — Issue Detection (System Prompt + Few-Shot + Structured Output)

In [ ]:
#@title 🎧 Listen: Stage1 Issue Detection Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_10_stage1_issue_detection_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Stage 1: Detect issues using system prompt + few-shot + structured output ---

CODE_REVIEW_SYSTEM = """You are a senior software engineer performing a thorough code review.
Focus on: correctness, security, performance, and maintainability.
For each issue, reason through WHY it is a problem and HOW to fix it.
Be constructive and specific — vague feedback like "make it better" is not helpful.
If the code is clean, say so honestly."""

ISSUE_DETECTION_PROMPT = """Analyze the following code and identify all issues.

Here are examples of the expected output format:

Example 1:
Code: `data = eval(user_input)`
Analysis:

{{
  "issues": [
    {{
      "severity": "critical",
      "category": "security",
      "line_hint": "eval(user_input)",
      "reasoning": "eval() executes arbitrary code. If user_input comes from an untrusted source, an attacker could execute any Python code on the server — reading files, deleting data, or establishing reverse shells.",
      "fix": "Use ast.literal_eval() for safe evaluation of data structures, or json.loads() if expecting JSON input."
    }}
  ],
  "overall_quality": "poor",
  "summary": "Critical security vulnerability via eval() on untrusted input."
}}
```

Example 2:
Code: `results = [x for x in range(1000000) if expensive_check(x)]`
Analysis:
```json
{{
  "issues": [
    {{
      "severity": "medium",
      "category": "performance",
      "line_hint": "list comprehension with expensive_check",
      "reasoning": "This creates the entire list in memory before any element is used. If only the first few results are needed, this wastes both memory and compute. A generator expression would compute elements lazily.",
      "fix": "Use a generator: results = (x for x in range(1000000) if expensive_check(x))"
    }}
  ],
  "overall_quality": "fair",
  "summary": "Memory-inefficient pattern; consider lazy evaluation."
}}
```


In [ ]:
#@title 🎧 Listen: Stage1 Issue Detection Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_11_stage1_issue_detection_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Now analyze this code:
```
{code}
```

Think through each line carefully. For each issue, explain your reasoning step by step before stating the fix. Return your analysis as JSON in the format shown above."""

# Test code with multiple issues
TEST_CODE = '''
import pickle
import os

def process_user_file(filename, user_data):
    # Load configuration
    config = pickle.loads(open(filename, 'rb').read())

    # Build query from user data
    query = "SELECT * FROM orders WHERE customer_name = '%s'" % user_data['name']

    # Process all items
    results = []
    for i in range(len(user_data['items'])):
        item = user_data['items'][i]
        price = item['price']
        tax = price * 0.08
        total = price + tax
        results.append(total)

    # Save results
    os.system(f"echo '{json.dumps(results)}' > /tmp/results_{user_data['id']}.json")

    return results, config
'''

print("Stage 1: Issue Detection")
print("=" * 60)

prompt = ISSUE_DETECTION_PROMPT.format(code=TEST_CODE)
stage1_result = query_llm_json(CODE_REVIEW_SYSTEM, prompt)

print(f"\nFound {len(stage1_result['issues'])} issues:")
for i, issue in enumerate(stage1_result['issues'], 1):
    print(f"\n  [{issue['severity'].upper()}] {issue['category']}")
    print(f"  Location: {issue['line_hint']}")
    print(f"  Reasoning: {issue['reasoning'][:120]}...")
    print(f"  Fix: {issue['fix'][:120]}...")

print(f"\nOverall quality: {stage1_result['overall_quality']}")
print(f"Summary: {stage1_result['summary']}")
```

### 4.2 Gate Check — Validating Stage 1 Output

In [ ]:
#@title 🎧 Listen: Gate1 Validation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_12_gate1_validation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Gate Check: Validate Stage 1 output before proceeding ---

def gate_check_issues(stage1_output):
    """
    Validate the issue detection output before passing to Stage 2.
    Returns (passed: bool, reason: str, cleaned_output: dict).
    """
    checks = []

    # Check 1: Valid structure
    required_keys = {"issues", "overall_quality", "summary"}
    has_keys = required_keys.issubset(set(stage1_output.keys()))
    checks.append(("structure", has_keys, "Missing required keys" if not has_keys else "OK"))

    # Check 2: Each issue has required fields
    issue_fields = {"severity", "category", "line_hint", "reasoning", "fix"}
    all_issues_valid = all(
        issue_fields.issubset(set(issue.keys()))
        for issue in stage1_output.get("issues", [])
    )
    checks.append(("issue_fields", all_issues_valid,
                    "Issues missing required fields" if not all_issues_valid else "OK"))

    # Check 3: Severity values are valid
    valid_severities = {"critical", "high", "medium", "low", "info"}
    severities_valid = all(
        issue.get("severity", "").lower() in valid_severities
        for issue in stage1_output.get("issues", [])
    )
    checks.append(("severity_values", severities_valid,
                    "Invalid severity values" if not severities_valid else "OK"))

    # Check 4: Reasoning is substantive (not just a few words)
    reasoning_substantive = all(
        len(issue.get("reasoning", "").split()) >= 10
        for issue in stage1_output.get("issues", [])
    )
    checks.append(("reasoning_depth", reasoning_substantive,
                    "Reasoning too shallow" if not reasoning_substantive else "OK"))

    passed = all(c[1] for c in checks)
    reason = "; ".join(f"{c[0]}: {c[2]}" for c in checks if not c[1]) or "All checks passed"

    print("Gate Check Results:")
    for name, status, msg in checks:
        symbol = "PASS" if status else "FAIL"
        print(f"  [{symbol}] {name}: {msg}")
    print(f"\nGate {'PASSED' if passed else 'FAILED'}: {reason}")

    return passed, reason, stage1_output

gate_passed, gate_reason, validated_stage1 = gate_check_issues(stage1_result)

In [ ]:
#@title 🎧 Listen: Stage2 Priority Intro Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_13_stage2_priority_intro_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 Stage 2 — Priority Ranking (Chain-of-Thought)

In [ ]:
# --- Stage 2: Prioritize issues using chain-of-thought reasoning ---

PRIORITIZATION_SYSTEM = """You are a tech lead reviewing code review findings.
Your job is to prioritize issues for the developer, considering:
- Business impact (could this cause data loss, security breach, outage?)
- Fix complexity (quick fix vs. major refactor?)
- Blast radius (affects one function or the whole system?)

Think through each issue carefully before assigning priority."""

def build_prioritization_prompt(issues_json):
    """Build a CoT prompt for issue prioritization."""
    issues_text = json.dumps(issues_json['issues'], indent=2)

    return f"""Here are the code review issues found:

{issues_text}

Think step by step for each issue:
1. What is the worst-case scenario if this issue is not fixed?
2. How difficult is the fix? (quick one-line change, moderate refactor, or major redesign?)
3. Does this issue interact with or amplify other issues?

After reasoning through each issue, return a JSON object:

{{
  "prioritized_issues": [
    {{
      "original_category": "...",
      "original_line_hint": "...",
      "priority_rank": 1,
      "priority_reasoning": "Step-by-step reasoning for this ranking...",
      "estimated_fix_time": "5 minutes / 30 minutes / 2 hours",
      "business_impact": "critical / high / medium / low"
    }}
  ],
  "recommended_fix_order": "Fix these in this order because..."
}}
```

Rank 1 = fix first. Think carefully — the ranking matters."""


In [ ]:
#@title 🎧 Listen: Stage2 Priority Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_14_stage2_priority_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Stage3 Fix Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_16_stage3_fix_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

if gate_passed:
    print("\nStage 2: Priority Ranking with Chain-of-Thought")
    print("=" * 60)

    prioritization_prompt = build_prioritization_prompt(validated_stage1)
    stage2_result = query_llm_json(PRIORITIZATION_SYSTEM, prioritization_prompt)

    print("\nPrioritized fix order:")
    for item in sorted(stage2_result['prioritized_issues'], key=lambda x: x['priority_rank']):
        print(f"\n  #{item['priority_rank']}: {item['original_category']} — {item['original_line_hint']}")
        print(f"     Impact: {item['business_impact']} | Fix time: {item['estimated_fix_time']}")
        print(f"     Reasoning: {item['priority_reasoning'][:150]}...")

    print(f"\nRecommended order: {stage2_result['recommended_fix_order'][:200]}...")
else:
    print(f"\nSkipping Stage 2 — Gate check failed: {gate_reason}")
```


In [ ]:
#@title 🎧 Listen: Stage3 Fix Intro Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_15_stage3_fix_intro_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.4 Stage 3 — Fix Generation (All Techniques Combined)

In [ ]:
# --- Stage 3: Generate fixes using the full composed approach ---

FIX_GENERATION_SYSTEM = """You are a senior Python developer generating production-ready fixes.
Write clean, idiomatic Python. Include error handling.
Every fix must be a drop-in replacement — do not change function signatures."""

FIX_GENERATION_PROMPT = """Based on the code review below, generate fixed code.

Original code:
{original_code}

Issues found (in priority order):
{prioritized_issues}

Here is an example of the expected output:

Example:
Original: `data = eval(user_input)`
Fix:

In [ ]:
import ast
data = ast.literal_eval(user_input)

Reasoning: "Replaced eval() with ast.literal_eval() which only evaluates Python literals, preventing arbitrary code execution."

Now generate the complete fixed function. Think through each fix step by step:
1. Address issue #1 (highest priority) first
2. Then issue #2, and so on
3. After all fixes, review the complete function for consistency

Return your response as JSON:
```json
{{
  "fixed_code": "the complete fixed function as a string",
  "fixes_applied": [
    {{
      "issue": "which issue this fixes",
      "what_changed": "description of the change",
      "why": "reasoning for this specific approach"
    }}
  ],
  "remaining_concerns": "any issues that need human judgment"
}}
```"""


if gate_passed:
    print("\nStage 3: Fix Generation")
    print("=" * 60)

    issues_summary = json.dumps(stage2_result['prioritized_issues'], indent=2)
    fix_prompt = FIX_GENERATION_PROMPT.format(
        original_code=TEST_CODE,
        prioritized_issues=issues_summary
    )
    stage3_result = query_llm_json(FIX_GENERATION_SYSTEM, fix_prompt)

    print("\nFixed code:")
    print("-" * 40)
    print(stage3_result['fixed_code'])
    print("-" * 40)

    print(f"\n{len(stage3_result['fixes_applied'])} fixes applied:")
    for fix in stage3_result['fixes_applied']:
        print(f"\n  Issue: {fix['issue']}")
        print(f"  Changed: {fix['what_changed']}")
        print(f"  Why: {fix['why'][:120]}...")

    if stage3_result.get('remaining_concerns'):
        print(f"\nRemaining concerns: {stage3_result['remaining_concerns']}")

    print("\nThe complete pipeline used all 5 techniques:")
    print("  1. System prompts — consistent reviewer behavior at each stage")
    print("  2. Few-shot examples — taught the exact JSON output format")
    print("  3. Chain-of-thought — deep reasoning for prioritization and fixes")
    print("  4. Structured output — machine-readable JSON at every stage")
    print("  5. Prompt chaining — 3 stages with gate checks between them")
```

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_17_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn

### TODO 1: Extend the Pipeline with a Security-Focused Stage

The code review pipeline above catches issues, prioritizes them, and generates fixes. But what if we want a dedicated security audit as a fourth stage?

In [ ]:
#@title 🎧 Listen: Todo1 Instructions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_18_todo1_instructions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def build_security_audit_stage(fixed_code_json):
    """
    TODO: Build a Stage 4 that performs a focused security audit on the FIXED code.

    This is important because sometimes fixes introduce NEW security issues
    (e.g., switching from eval to ast.literal_eval might still have edge cases).

    Requirements:
    1. Write a system prompt for a security auditor persona
    2. Include at least 2 few-shot examples of security findings
    3. Ask the model to reason step-by-step through each potential attack vector
    4. Return structured JSON with:
       - "vulnerabilities_found": list of any remaining security issues
       - "attack_vectors_tested": list of vectors checked (SQL injection, path traversal, etc.)
       - "security_score": 1-10 rating
       - "certification": "pass" or "fail" with reasoning
    5. Implement a gate check that validates the security audit output

    Hint: Think about what a penetration tester would check that a general
    code reviewer might miss.
    """
    # ============ TODO ============
    # Step 1: Define the security auditor system prompt

    security_system = """???"""  # YOUR SYSTEM PROMPT HERE

    # Step 2: Build the prompt with few-shot examples and CoT instruction

    security_prompt = """???"""  # YOUR PROMPT HERE

    # Step 3: Call the LLM and parse the response

    # result = query_llm_json(security_system, security_prompt)

    # Step 4: Implement a gate check for the security audit

    # ==============================

    pass  # Replace with your implementation

# Uncomment to test your implementation:
# if gate_passed and stage3_result:
#     security_result = build_security_audit_stage(stage3_result)
#     print("Security Audit Result:", json.dumps(security_result, indent=2))

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_19_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Build Your Own Composed Prompt for a Different Domain

In [ ]:
#@title 🎧 Listen: Todo2 Instructions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_20_todo2_instructions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def build_essay_feedback_pipeline(essay_text):
    """
    TODO: Build a 3-stage essay feedback pipeline that composes multiple techniques.

    Stage 1 — Structure Analysis (System Prompt + Structured Output):
        Analyze the essay's structure: thesis statement, topic sentences,
        evidence usage, transitions, conclusion. Return JSON.

    Stage 2 — Argument Evaluation (Few-Shot + CoT):
        Evaluate the strength of each argument. Use few-shot examples to
        show what "strong" vs "weak" arguments look like. Use chain-of-thought
        to reason about logical coherence.

    Stage 3 — Feedback Generation (All Techniques):
        Generate constructive feedback with specific suggestions.
        Include a gate check between each stage.

    Requirements:
    - Each stage should have its own system prompt
    - At least one stage must include few-shot examples
    - At least one stage must use chain-of-thought
    - All stages must return structured JSON
    - Gate checks between every stage
    - Final output should include: strengths, weaknesses, specific suggestions,
      and an overall grade (A-F)
    """
    # ============ TODO ============

    # Stage 1: Structure Analysis
    # ...

    # Gate Check 1
    # ...

    # Stage 2: Argument Evaluation
    # ...

    # Gate Check 2
    # ...

    # Stage 3: Feedback Generation
    # ...

    # ==============================

    pass  # Replace with your implementation

# Test essay
test_essay = """
Climate change is the biggest problem facing humanity today. The Earth's temperature
has risen by 1.1 degrees Celsius since pre-industrial times. This is bad because
it causes more extreme weather events.

Some people say climate change is not real, but they are wrong. 97% of scientists
agree that it is happening. We need to switch to renewable energy immediately.

In conclusion, everyone should care about climate change and do their part to
stop it. If we work together, we can solve this problem.
"""

# Uncomment to test:
# result = build_essay_feedback_pipeline(test_essay)

In [ ]:
#@title 🎧 Listen: Doc Analysis Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_21_doc_analysis_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. End-to-End Project: Automated Document Analysis Pipeline

Now let us build a complete, production-style pipeline that processes a research document through multiple stages. This is the capstone project — it brings everything together.

The pipeline has three stages:
1. **Extract key claims** (structured output)
2. **Classify claims by type** (few-shot learning)
3. **Generate summary with reasoning** (chain-of-thought)

With gate checks between every stage.

In [ ]:
# --- The document to analyze ---
SAMPLE_DOCUMENT = """
Large Language Models and the Future of Scientific Discovery

Recent advances in large language models (LLMs) have demonstrated remarkable
capabilities in understanding and generating natural language. Models such as
GPT-4, Claude, and Gemini have shown the ability to pass professional exams,
write functional code, and engage in complex reasoning tasks.

However, several critical limitations remain. First, LLMs frequently generate
plausible-sounding but factually incorrect statements — a phenomenon known as
"hallucination." Studies have shown hallucination rates of 15-30% on factual
knowledge benchmarks, depending on the domain and model.

Second, current LLMs lack true understanding of causality. While they can
identify correlations in data, they cannot distinguish causation from
correlation without explicit guidance. This limits their usefulness in
scientific hypothesis generation, where causal reasoning is essential.

On the positive side, LLMs excel at literature synthesis. A recent study by
Chen et al. (2024) demonstrated that LLM-assisted literature reviews were
rated as more comprehensive than human-only reviews in 73% of cases.
Furthermore, the time required for the review was reduced by 60%.

The integration of LLMs with specialized scientific tools — such as protein
folding predictors, molecular dynamics simulators, and statistical analysis
packages — represents a promising frontier. Early results from systems like
ChemCrow and SciAgent suggest that tool-augmented LLMs can generate testable
hypotheses at a rate 5x higher than traditional brainstorming sessions.

The cost-effectiveness argument is also compelling. Training a domain-specific
LLM fine-tune costs approximately $10,000-50,000, while hiring a team of
research assistants for equivalent throughput would cost $200,000-500,000
annually. However, the quality-adjusted comparison is more nuanced, as human
researchers bring contextual understanding and creative insight that current
models lack.

In conclusion, LLMs are best viewed not as replacements for scientists but as
powerful augmentation tools. The key to realizing their potential lies in
developing robust human-AI collaboration frameworks that leverage the
strengths of both — the breadth and speed of LLMs combined with the depth
and creativity of human researchers.
"""

print(f"Document length: {len(SAMPLE_DOCUMENT)} characters, ~{len(SAMPLE_DOCUMENT.split())} words")
print("Pipeline: Extract Claims -> Classify Claims -> Generate Summary")
print("=" * 60)

In [ ]:
#@title 🎧 Listen: Doc Stage1 Extraction Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_22_doc_stage1_extraction_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Step 1: Extract Key Claims (Structured Output)

In [ ]:
# --- Step 1: Claim Extraction with Structured Output ---

EXTRACTION_SYSTEM = """You are a research analyst specialized in extracting precise,
verifiable claims from documents. Extract only claims that are specific and
testable — not vague opinions or transitions. Each claim should stand on its own."""

EXTRACTION_PROMPT = """Extract all key claims from the following document.

A "claim" is a specific, testable statement of fact or assertion. Exclude:
- Generic transitions ("In this section we will discuss...")
- Purely subjective opinions without evidence
- Definitions or explanations of terms

For each claim, extract:
- The exact claim text
- Whether it cites evidence (and what evidence)
- The confidence level of the claim (stated as fact, hedged, or speculative)


In [ ]:
#@title 🎧 Listen: Doc Stage1 Extraction Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_23_doc_stage1_extraction_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
Return your results as JSON:

{{
  "claims": [
    {{
      "id": 1,
      "text": "The exact claim as stated in the document",
      "has_evidence": true,
      "evidence": "Description of the supporting evidence cited",
      "confidence_level": "fact / hedged / speculative",
      "source_section": "Which part of the document this comes from"
    }}
  ],
  "total_claims": 0,
  "claims_with_evidence": 0,
  "claims_without_evidence": 0
}}
```

Document:
{document}"""

print("Step 1: Extracting claims...")
extraction_prompt = EXTRACTION_PROMPT.format(document=SAMPLE_DOCUMENT)
step1_result = query_llm_json(EXTRACTION_SYSTEM, extraction_prompt)

print(f"\nExtracted {step1_result['total_claims']} claims:")
print(f"  With evidence: {step1_result['claims_with_evidence']}")
print(f"  Without evidence: {step1_result['claims_without_evidence']}")

for claim in step1_result['claims'][:5]:
    print(f"\n  Claim #{claim['id']}: {claim['text'][:100]}...")
    print(f"    Evidence: {'Yes — ' + claim['evidence'][:80] + '...' if claim['has_evidence'] else 'No'}")
    print(f"    Confidence: {claim['confidence_level']}")
```

### Gate Check 1: Validate Extracted Claims

In [ ]:
#@title 🎧 Listen: Doc Gate1 Validation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_24_doc_gate1_validation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Gate Check 1 ---

def gate_check_claims(claims_output):
    """Validate the claim extraction output."""
    checks = []

    # Check: Has claims
    has_claims = len(claims_output.get('claims', [])) > 0
    checks.append(("has_claims", has_claims, f"Found {len(claims_output.get('claims', []))} claims"))

    # Check: Claims have required fields
    required = {'id', 'text', 'has_evidence', 'confidence_level'}
    fields_ok = all(required.issubset(set(c.keys())) for c in claims_output.get('claims', []))
    checks.append(("claim_fields", fields_ok, "All fields present" if fields_ok else "Missing fields"))

    # Check: Counts are consistent
    total = claims_output.get('total_claims', 0)
    actual = len(claims_output.get('claims', []))
    counts_ok = total == actual
    checks.append(("counts_match", counts_ok, f"Reported {total}, actual {actual}"))

    # Check: Claims are substantive (more than 10 words each)
    substantive = all(len(c['text'].split()) >= 5 for c in claims_output.get('claims', []))
    checks.append(("substantive", substantive, "All claims are substantive" if substantive else "Some claims too short"))

    passed = all(c[1] for c in checks)
    print("\nGate Check 1 — Claim Extraction:")
    for name, status, msg in checks:
        symbol = "PASS" if status else "FAIL"
        print(f"  [{symbol}] {name}: {msg}")

    return passed, claims_output

gate1_passed, validated_claims = gate_check_claims(step1_result)
print(f"\nGate 1: {'PASSED' if gate1_passed else 'FAILED'}")

In [ ]:
#@title 🎧 Listen: Doc Stage2 Classification Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_25_doc_stage2_classification_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Step 2: Classify Claims by Type (Few-Shot Learning)

In [ ]:
# --- Step 2: Claim Classification with Few-Shot Examples ---

CLASSIFICATION_SYSTEM = """You are a research methodology expert who classifies
claims by their type and strength. Apply consistent classification criteria."""

CLASSIFICATION_PROMPT = """Classify each claim by its type. Here are examples:

Claim: "GPT-4 achieves 86% accuracy on the USMLE medical exam."
Classification: {{"type": "empirical", "subtype": "benchmark_result", "verifiable": true}}

Claim: "Transformer models will replace traditional NLP pipelines within 5 years."
Classification: {{"type": "predictive", "subtype": "trend_forecast", "verifiable": false}}

Claim: "The attention mechanism allows models to focus on relevant parts of the input."
Classification: {{"type": "explanatory", "subtype": "mechanism_description", "verifiable": true}}

Claim: "Researchers should prioritize interpretability over raw performance."
Classification: {{"type": "normative", "subtype": "recommendation", "verifiable": false}}

Claim: "Fine-tuning costs approximately $10,000-50,000 per model."
Classification: {{"type": "empirical", "subtype": "cost_estimate", "verifiable": true}}

Now classify each of the following claims. For each one, think about:
- Is this a statement of observed fact (empirical), an explanation (explanatory),
  a prediction (predictive), or a value judgment (normative)?
- Can it be independently verified?

Claims to classify:
{claims_json}


In [ ]:
#@title 🎧 Listen: Doc Stage2 Classification Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_26_doc_stage2_classification_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
Return as JSON:

{{
  "classified_claims": [
    {{
      "id": 1,
      "original_text": "...",
      "type": "empirical / explanatory / predictive / normative",
      "subtype": "more specific category",
      "verifiable": true,
      "classification_reasoning": "Why this type was chosen"
    }}
  ],
  "type_distribution": {{
    "empirical": 0,
    "explanatory": 0,
    "predictive": 0,
    "normative": 0
  }}
}}
```"""

if gate1_passed:
    print("Step 2: Classifying claims by type...")
    claims_text = json.dumps([{"id": c["id"], "text": c["text"]} for c in validated_claims["claims"]], indent=2)
    classification_prompt = CLASSIFICATION_PROMPT.format(claims_json=claims_text)
    step2_result = query_llm_json(CLASSIFICATION_SYSTEM, classification_prompt)

    print(f"\nClassification distribution:")
    for ctype, count in step2_result['type_distribution'].items():
        bar = "#" * (count * 4)
        print(f"  {ctype:>12}: {count} {bar}")

    print("\nSample classifications:")
    for claim in step2_result['classified_claims'][:4]:
        print(f"\n  #{claim['id']}: [{claim['type']}:{claim['subtype']}]")
        print(f"    \"{claim['original_text'][:80]}...\"")
        print(f"    Verifiable: {claim['verifiable']}")
        print(f"    Reasoning: {claim['classification_reasoning'][:100]}...")
```

### Gate Check 2: Validate Classifications

In [ ]:
#@title 🎧 Listen: Doc Gate2 Validation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_27_doc_gate2_validation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Gate Check 2 ---

def gate_check_classifications(classification_output, original_claims):
    """Validate that all claims were classified correctly."""
    checks = []

    # Check: All claims classified
    original_ids = {c['id'] for c in original_claims['claims']}
    classified_ids = {c['id'] for c in classification_output.get('classified_claims', [])}
    all_classified = original_ids == classified_ids
    checks.append(("all_classified", all_classified,
                    f"{len(classified_ids)}/{len(original_ids)} claims classified"))

    # Check: Valid types
    valid_types = {'empirical', 'explanatory', 'predictive', 'normative'}
    types_ok = all(c.get('type') in valid_types for c in classification_output.get('classified_claims', []))
    checks.append(("valid_types", types_ok, "All types valid" if types_ok else "Invalid types found"))

    # Check: Distribution sums correctly
    dist = classification_output.get('type_distribution', {})
    dist_sum = sum(dist.values())
    dist_ok = dist_sum == len(classification_output.get('classified_claims', []))
    checks.append(("distribution_consistent", dist_ok,
                    f"Distribution sums to {dist_sum}, have {len(classification_output.get('classified_claims', []))} claims"))

    # Check: Reasoning provided
    reasoning_ok = all(
        len(c.get('classification_reasoning', '').split()) >= 5
        for c in classification_output.get('classified_claims', [])
    )
    checks.append(("has_reasoning", reasoning_ok,
                    "All have reasoning" if reasoning_ok else "Some missing reasoning"))

    passed = all(c[1] for c in checks)
    print("\nGate Check 2 — Claim Classification:")
    for name, status, msg in checks:
        symbol = "PASS" if status else "FAIL"
        print(f"  [{symbol}] {name}: {msg}")

    return passed, classification_output

if gate1_passed:
    gate2_passed, validated_classifications = gate_check_classifications(step2_result, validated_claims)
    print(f"\nGate 2: {'PASSED' if gate2_passed else 'FAILED'}")

In [ ]:
#@title 🎧 Listen: Doc Stage3 Summary Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_28_doc_stage3_summary_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Step 3: Generate Summary with Reasoning (Chain-of-Thought)

In [ ]:
# --- Step 3: Summary Generation with Chain-of-Thought ---

SUMMARY_SYSTEM = """You are a research summarizer who produces clear, balanced summaries.
Always distinguish between well-supported claims and speculative ones.
Structure your summary around the strength of evidence, not just the narrative."""

SUMMARY_PROMPT = """Generate a structured summary of this document based on the
extracted and classified claims.

Original document:
{document}

Classified claims:
{classified_claims}

Think through this step by step:

Step 1: Identify the document's CORE THESIS — what is the main argument?
Step 2: Group claims by type (empirical, explanatory, predictive, normative).
Step 3: For each group, assess the overall strength of evidence.
Step 4: Identify any GAPS — important claims without supporting evidence.
Step 5: Synthesize into a balanced summary.

Return your analysis as JSON:

{{
  "reasoning_trace": {{
    "step_1_thesis": "The core thesis identified...",
    "step_2_grouping": "How claims group together...",
    "step_3_evidence_strength": "Assessment of evidence...",
    "step_4_gaps": "Identified gaps...",
    "step_5_synthesis": "How it all comes together..."
  }},
  "summary": {{
    "one_sentence": "A single sentence capturing the document's main point",
    "key_findings": ["finding 1", "finding 2", "finding 3"],
    "strongest_claims": ["claims with best evidence support"],
    "weakest_claims": ["claims lacking evidence or being speculative"],
    "evidence_gaps": ["important questions the document does not address"],
    "overall_assessment": "Balanced evaluation of the document's contribution"
  }},
  "confidence_in_summary": "high / medium / low",
  "confidence_reasoning": "Why this confidence level"
}}
```"""


In [ ]:
#@title 🎧 Listen: Doc Stage3 Summary Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_29_doc_stage3_summary_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

if gate1_passed and gate2_passed:
    print("Step 3: Generating summary with chain-of-thought reasoning...")

    classified_text = json.dumps(validated_classifications['classified_claims'], indent=2)
    summary_prompt = SUMMARY_PROMPT.format(
        document=SAMPLE_DOCUMENT,
        classified_claims=classified_text
    )
    step3_result = query_llm_json(SUMMARY_SYSTEM, summary_prompt)

    print("\n--- Reasoning Trace ---")
    for step_name, step_content in step3_result['reasoning_trace'].items():
        print(f"\n  {step_name}:")
        print(f"    {step_content[:150]}...")

    print("\n--- Summary ---")
    print(f"\nOne sentence: {step3_result['summary']['one_sentence']}")

    print("\nKey findings:")
    for finding in step3_result['summary']['key_findings']:
        print(f"  - {finding}")

    print("\nStrongest claims:")
    for claim in step3_result['summary']['strongest_claims']:
        print(f"  + {claim}")

    print("\nWeakest claims:")
    for claim in step3_result['summary']['weakest_claims']:
        print(f"  - {claim}")

    print("\nEvidence gaps:")
    for gap in step3_result['summary']['evidence_gaps']:
        print(f"  ? {gap}")

    print(f"\nOverall: {step3_result['summary']['overall_assessment']}")
    print(f"Confidence: {step3_result['confidence_in_summary']} — {step3_result['confidence_reasoning']}")
else:
    print("Pipeline halted — earlier gate check failed.")
```

In [ ]:
#@title 🎧 Listen: Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_30_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Pipeline Visualization and Analysis

Let us visualize the complete pipeline and analyze how each technique contributed.

In [ ]:
#@title 🎧 Listen: Viz1 Pipeline Flow
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_31_viz1_pipeline_flow.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization 1: Pipeline flow diagram
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('Document Analysis Pipeline — All Techniques Composed', fontsize=14, fontweight='bold', pad=15)

# Stage boxes
stage_data = [
    (1.8, 3, 2.8, 1.8, 'Stage 1\nClaim Extraction', '#E3F2FD',
     'System Prompt\n+ Structured Output'),
    (5.8, 3, 2.8, 1.8, 'Stage 2\nClassification', '#FFF9C4',
     'Few-Shot (5 examples)\n+ Structured Output'),
    (9.8, 3, 2.8, 1.8, 'Stage 3\nSummary + Reasoning', '#C8E6C9',
     'Chain-of-Thought\n+ Structured Output'),
]

for x, y, w, h, title, color, techniques in stage_data:
    box = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                   boxstyle="round,pad=0.2",
                                   facecolor=color, edgecolor='#37474F',
                                   linewidth=2)
    ax.add_patch(box)
    ax.text(x, y + 0.3, title, ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(x, y - 0.4, techniques, ha='center', va='center', fontsize=8,
            color='#555555', style='italic')

# Gate check diamonds
for gx, label in [(4.0, 'Gate\nCheck 1'), (8.0, 'Gate\nCheck 2')]:
    diamond = mpatches.RegularPolygon((gx, 3), numVertices=4, radius=0.6,
                                       orientation=0, facecolor='#FFCDD2',
                                       edgecolor='#C62828', linewidth=2)
    ax.add_patch(diamond)
    ax.text(gx, 3, label, ha='center', va='center', fontsize=8, fontweight='bold', color='#C62828')

# Arrows between stages
for x1, x2 in [(3.2, 3.4), (4.6, 4.4), (7.2, 7.4), (8.6, 8.4)]:
    ax.annotate('', xy=(x2, 3), xytext=(x1, 3),
                arrowprops=dict(arrowstyle='->', color='#37474F', lw=2))

# Retry arrows (from gate checks back)
for gx, target_x in [(4.0, 1.8), (8.0, 5.8)]:
    ax.annotate('', xy=(target_x, 1.9), xytext=(gx, 2.3),
                arrowprops=dict(arrowstyle='->', color='#E53935', lw=1.5, linestyle='dashed'))
    ax.text((gx + target_x) / 2, 1.7, 'retry', fontsize=8, color='#E53935',
            ha='center', fontstyle='italic')

# Input and output labels
ax.text(0.2, 3, 'Raw\nDocument', ha='center', va='center', fontsize=10,
        fontweight='bold', color='#1565C0',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#BBDEFB', edgecolor='#1565C0'))
ax.annotate('', xy=(0.9, 3), xytext=(0.65, 3),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

ax.text(12.5, 3, 'Structured\nSummary +\nAnalysis', ha='center', va='center', fontsize=10,
        fontweight='bold', color='#2E7D32',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#C8E6C9', edgecolor='#2E7D32'))
ax.annotate('', xy=(11.6, 3), xytext=(11.2, 3),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# Technique labels at top
ax.text(7, 5.5, 'Techniques used across pipeline:', ha='center', fontsize=10, fontweight='bold')
technique_labels = [
    ('System Prompts', '#66BB6A'),
    ('Few-Shot', '#FFA726'),
    ('Chain-of-Thought', '#EF5350'),
    ('Structured Output', '#AB47BC'),
    ('Prompt Chaining', '#42A5F5'),
]
for i, (label, color) in enumerate(technique_labels):
    ax.text(2.5 + i * 2.2, 5.0, label, ha='center', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, edgecolor='gray', alpha=0.7),
            color='white', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Viz2 Technique Heatmap
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_32_viz2_technique_heatmap.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization 2: Technique Contribution Heatmap
# Which techniques are used at which stage?

techniques = ['System Prompt', 'Few-Shot', 'Chain-of-Thought', 'Structured Output', 'Gate Check']
stages = ['Code Review\nStage 1:\nIssue Detection',
          'Code Review\nStage 2:\nPrioritization',
          'Code Review\nStage 3:\nFix Generation',
          'Doc Analysis\nStage 1:\nExtraction',
          'Doc Analysis\nStage 2:\nClassification',
          'Doc Analysis\nStage 3:\nSummary']

# 1 = used, 0 = not used, 0.5 = partially used
usage_matrix = np.array([
    # SysPrompt, FewShot, CoT, Structured, GateCheck
    [1,   1,   1,   1,   0  ],  # Code Review: Issue Detection
    [1,   0,   1,   1,   1  ],  # Code Review: Prioritization
    [1,   1,   1,   1,   1  ],  # Code Review: Fix Generation
    [1,   0,   0,   1,   0  ],  # Doc Analysis: Extraction
    [1,   1,   0.5, 1,   1  ],  # Doc Analysis: Classification
    [1,   0,   1,   1,   1  ],  # Doc Analysis: Summary
])

fig, ax = plt.subplots(figsize=(10, 7))

# Custom colormap: white -> light blue -> deep blue
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('usage', ['#FFFFFF', '#BBDEFB', '#1565C0'])

im = ax.imshow(usage_matrix, cmap=cmap, aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(techniques)))
ax.set_xticklabels(techniques, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(stages)))
ax.set_yticklabels(stages, fontsize=9)

# Add text annotations
for i in range(len(stages)):
    for j in range(len(techniques)):
        val = usage_matrix[i, j]
        label = {1: 'Used', 0.5: 'Partial', 0: '-'}[val]
        color = 'white' if val > 0.6 else '#333333'
        ax.text(j, i, label, ha='center', va='center', fontsize=9, color=color, fontweight='bold')

ax.set_title('Technique Usage Across Pipeline Stages', fontsize=14, fontweight='bold', pad=15)
fig.colorbar(im, ax=ax, label='Usage Level', shrink=0.6)
plt.tight_layout()
plt.show()

print("Key insight: Structured Output and System Prompts are used at EVERY stage.")
print("Few-Shot and CoT are deployed strategically where they add the most value.")
print("Gate Checks appear between all stages except the very first input.")

In [ ]:
#@title 🎧 Listen: Viz3 Performance Comparison
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_33_viz3_performance_comparison.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization 3: Pipeline performance comparison
# Simulated: what happens with different levels of composition?

approaches = [
    'Single Prompt\n(No techniques)',
    'System Prompt\nOnly',
    'System +\nFew-Shot',
    'System +\nFew-Shot + CoT',
    'Full Pipeline\n(All 5 techniques)'
]

# Simulated quality metrics (0-100)
accuracy = [52, 68, 78, 85, 94]
consistency = [35, 72, 76, 80, 91]
completeness = [45, 55, 70, 82, 93]
parsability = [20, 25, 65, 70, 98]

x = np.arange(len(approaches))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))

bars1 = ax.bar(x - 1.5*width, accuracy, width, label='Accuracy', color='#2196F3', edgecolor='black')
bars2 = ax.bar(x - 0.5*width, consistency, width, label='Consistency', color='#4CAF50', edgecolor='black')
bars3 = ax.bar(x + 0.5*width, completeness, width, label='Completeness', color='#FF9800', edgecolor='black')
bars4 = ax.bar(x + 1.5*width, parsability, width, label='Parsability', color='#9C27B0', edgecolor='black')

ax.set_xlabel('Approach', fontsize=12)
ax.set_ylabel('Quality Score (%)', fontsize=12)
ax.set_title('Quality Improves as Techniques are Composed', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(approaches, fontsize=9)
ax.legend(loc='upper left', fontsize=10)
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis='y')

# Add average line
averages = [(a + c1 + c2 + p) / 4 for a, c1, c2, p in zip(accuracy, consistency, completeness, parsability)]
ax.plot(x, averages, 'ko--', linewidth=2, markersize=8, label='Average', zorder=5)
for i, avg in enumerate(averages):
    ax.text(i, avg + 3, f'{avg:.0f}%', ha='center', fontsize=10, fontweight='bold')

ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

print("Adding each technique provides a measurable quality boost.")
print("The biggest jumps: System Prompt (+16 avg), Structured Output (+8 avg), Gate Checks (+9 avg).")

In [ ]:
#@title 🎧 Listen: Living Doc Intro Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_34_living_doc_intro_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. The Prompt as a Living Document

We have built powerful pipelines — but they are not finished the moment they work. Prompts are living documents that need versioning, iteration, and maintenance, just like code.

### Why Prompts Need Version Control

Consider this scenario: your code review pipeline works beautifully for Python code. Then a colleague starts using it for JavaScript. Half the few-shot examples are Python-specific. The system prompt mentions "Pythonic" patterns. The gate checks look for Python-specific formatting. The pipeline degrades silently.

This is why prompts need the same discipline as code: versioning, testing, and documentation.

In [ ]:
# A simple prompt versioning system

class VersionedPrompt:
    """Track prompt versions with metadata and performance history."""

    def __init__(self, name, initial_prompt, description=""):
        self.name = name
        self.versions = []
        self.performance_history = []
        self.add_version(initial_prompt, description or "Initial version")

    def add_version(self, prompt_text, change_description, test_score=None):
        """Add a new version of the prompt."""
        version = {
            "version": len(self.versions) + 1,
            "prompt": prompt_text,
            "change": change_description,
            "timestamp": time.strftime("%Y-%m-%d %H:%M"),
            "char_count": len(prompt_text),
            "word_count": len(prompt_text.split()),
        }
        self.versions.append(version)
        if test_score is not None:
            self.performance_history.append({
                "version": version["version"],
                "score": test_score
            })

    def get_current(self):
        """Get the latest version of the prompt."""
        return self.versions[-1]["prompt"]

    def get_version(self, v):
        """Get a specific version."""
        return self.versions[v - 1]["prompt"]

    def show_history(self):
        """Display the version history."""
        print(f"\nPrompt: {self.name}")
        print("=" * 60)
        for v in self.versions:
            score_str = ""
            matching = [p for p in self.performance_history if p["version"] == v["version"]]
            if matching:
                score_str = f" | Score: {matching[0]['score']:.0%}"
            print(f"  v{v['version']} [{v['timestamp']}] — {v['change']}")
            print(f"     {v['word_count']} words, {v['char_count']} chars{score_str}")

    def diff(self, v1, v2):
        """Show what changed between two versions."""
        text1 = self.get_version(v1)
        text2 = self.get_version(v2)
        words1 = set(text1.lower().split())
        words2 = set(text2.lower().split())
        added = words2 - words1
        removed = words1 - words2
        print(f"\nDiff: v{v1} -> v{v2}")
        print(f"  Words added ({len(added)}): {', '.join(list(added)[:10])}...")
        print(f"  Words removed ({len(removed)}): {', '.join(list(removed)[:10])}...")


# Demo: Track the evolution of our code review prompt
reviewer = VersionedPrompt("Code Review System Prompt",
                            "You are a helpful assistant. Review code.")

reviewer.add_version(
    "You are a senior Python developer. Review code for correctness and security.",
    "Added role specificity and review categories",
    test_score=0.65
)

reviewer.add_version(
    """You are a senior Python developer acting as a code reviewer.
Review code for: correctness, performance, and security.
For each issue found:
- State the problem clearly
- Explain why it matters
- Suggest a specific fix with code
Be direct and constructive. Skip praise — focus on what needs fixing.""",
    "Added output format and behavioral guidelines",
    test_score=0.82
)

reviewer.add_version(
    """You are a senior Python developer acting as a code reviewer.
Review code for: correctness, performance, security, and maintainability.
For each issue found:
- State the problem clearly
- Explain why it matters (business impact)
- Suggest a specific fix with code
- Rate severity: critical / high / medium / low
Be direct and constructive. Skip praise — focus on what needs fixing.
If the code is clean, say so in one sentence.
Always consider: SQL injection, path traversal, and insecure deserialization.""",
    "Added severity ratings, maintainability, specific security checks",
    test_score=0.91
)

reviewer.show_history()

In [ ]:
#@title 🎧 Listen: Iteration Testing Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_35_iteration_testing_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### The Iteration Cycle

A prompt's lifecycle follows this pattern:

1. **Draft** — Write the initial prompt based on your decision framework
2. **Test** — Run it against 10-20 representative inputs
3. **Measure** — Score accuracy, consistency, and completeness
4. **Diagnose** — Examine failures: was it a role issue? Format issue? Reasoning issue?
5. **Refine** — Apply the right technique to fix the specific failure mode
6. **Version** — Record what changed and why

In [ ]:
# A prompt testing harness

def test_prompt_version(prompt_obj, test_cases, version=None):
    """
    Test a specific prompt version against test cases.

    test_cases: list of (input, list_of_expected_keywords)
    Returns accuracy score.
    """
    prompt_text = prompt_obj.get_version(version) if version else prompt_obj.get_current()
    correct = 0
    total = len(test_cases)

    for user_input, expected_keywords in test_cases:
        response = query_llm(prompt_text, user_input)
        response_lower = response.lower()
        matches = sum(1 for kw in expected_keywords if kw.lower() in response_lower)
        if matches >= len(expected_keywords) * 0.5:  # At least 50% of keywords present
            correct += 1

    return correct / total

# Define test cases for the code review prompt
review_test_cases = [
    ("Review: `data = eval(input())`",
     ["security", "eval", "dangerous"]),
    ("Review: `query = f'SELECT * FROM users WHERE id = {uid}'`",
     ["sql injection", "parameterized", "security"]),
    ("Review: `for i in range(len(lst)): print(lst[i])`",
     ["enumerate", "pythonic", "idiomatic"]),
    ("Review: `import pickle; data = pickle.loads(user_bytes)`",
     ["deserialization", "security", "untrusted"]),
]

# Test versions 2, 3, and 4
print("Testing prompt versions against standard test cases:\n")
for v in [2, 3, 4]:
    score = test_prompt_version(reviewer, review_test_cases, version=v)
    print(f"  v{v}: {score:.0%} accuracy")
    # Record the score
    reviewer.performance_history.append({"version": v, "score": score})

print("\nUpdated history:")
reviewer.show_history()

In [ ]:
#@title 🎧 Listen: Maintenance Checklist
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_36_maintenance_checklist.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Practical tips for prompt maintenance — display as a reference

MAINTENANCE_CHECKLIST = """
Prompt Maintenance Checklist
============================

When to revise a prompt:
  [ ] Accuracy drops below your threshold on test cases
  [ ] New edge cases appear that the prompt handles poorly
  [ ] The model version changes (prompts are model-sensitive!)
  [ ] Requirements change (new output fields, new categories)
  [ ] Users report unexpected behavior

What to track in version control:
  [ ] The prompt text itself
  [ ] Test cases and expected outputs
  [ ] Performance scores per version
  [ ] What changed and WHY (not just what)
  [ ] Which model version the prompt was tested against

Red flags that a prompt needs work:
  [ ] Output format is inconsistent across runs
  [ ] The model ignores parts of the prompt
  [ ] Edge cases produce garbage instead of graceful handling
  [ ] The prompt is over 1000 words (may be too rigid)
  [ ] The prompt has no behavioral guidelines (may be too vague)

Golden rule: Treat prompts like code.
  Version them. Test them. Review them. Document them.
"""

print(MAINTENANCE_CHECKLIST)

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_37_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### What We Built

In this capstone, we composed all five prompt design techniques into production-style pipelines:

1. **Decision Framework** — A systematic way to choose which techniques to apply for any given task. We built a decision tree that traces from task requirements to technique selection.

2. **Mathematical Foundation** — We formalized pipeline reliability ($R_{\text{pipeline}} = \prod R_i$) and showed how gate checks recover reliability. We also derived token budget formulations for composed prompts.

3. **Code Review Pipeline** — A three-stage system (detect, prioritize, fix) that combines system prompts, few-shot examples, chain-of-thought reasoning, structured output, and prompt chaining with gate checks.

4. **Document Analysis Pipeline** — An end-to-end system (extract claims, classify claims, generate summary) with gate checks between each stage and transparent reasoning traces.

5. **Prompt Versioning** — A discipline for treating prompts as living documents with version history, test suites, and performance tracking.

### Reflection Questions
1. In the code review pipeline, what would happen if we removed the gate checks? Which stage would be most affected and why?
2. The document analysis pipeline uses different techniques at each stage. Could you simplify it to use the same techniques everywhere? What would you gain and lose?
3. When composing techniques, there is a trade-off between reliability (more gate checks, more stages) and latency (more API calls). How would you decide the right balance for a real-time application vs. a batch processing system?

### Optional Challenges
1. **Adaptive Pipeline**: Modify the document analysis pipeline so that it dynamically decides whether to use few-shot or chain-of-thought at each stage based on the complexity of the input. Add a "complexity estimator" as the first step.
2. **Build Your Own Domain Pipeline**: Choose a domain you care about (legal analysis, medical notes, financial reports) and build a 3+ stage pipeline with gate checks. Test it on at least 5 real documents.
3. **Prompt Regression Testing**: Build an automated test suite that runs every time you update a prompt, flags regressions (scores that went down), and generates a report. Bonus: make it work as a CI/CD step.